# HUS-QA: Quality Assurance for Dental X-ray Segmentation without Ground Truth

Code for the paper **"Beyond Ground Truth: A Heuristic Unsupervised Quality Assurance Engine for Diagnosing Calibrational Blindness in Dental Segmentation"** (IEEE CSDE 2026).

The notebook follows the three stages of the paper (Fig. 1):

| Stage | What it does | Paper section |
|---|---|---|
| 1 | SimCLR pre-training of a ResNet-18 encoder on unlabeled periapical X-rays | III-B |
| 2 | U-Net segmentation, with the SimCLR encoder as the starting point | III-C |
| 3 | HUS-QA: a 0-100 quality score for each predicted mask, computed **without** the ground-truth mask | III-D |

After the three stages, the notebook reproduces Tables I-III and Figures 2-5 of the paper.

**How to run:** run the cells from top to bottom (a GPU is recommended). A training stage is skipped automatically if its weight file already exists.

## 0. Setup

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset

# In Colab, mount Google Drive. Outside Colab this step is skipped.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

In [ ]:
# ---------- Paths: change these to match your setup ----------
PROJECT_DIR = Path("/content/drive/MyDrive/Dental_QA_Project")
UNLABELED_DIR = PROJECT_DIR / "dataset" / "Unlabeled_Pretrain" / "images"   # 2,500 unlabeled X-rays (Stage 1)
LABELED_DIR = PROJECT_DIR / "dataset" / "Labeled_Downstream"     # train_*, valid_*, test_* folders (Stage 2)
RESULTS_DIR = PROJECT_DIR / "results"                            # figures and tables are saved here

SSL_ENCODER_PATH = PROJECT_DIR / "simclr_encoder_resnet18_epoch100.pth"   # output of Stage 1
SSL_RESUME_PATH = PROJECT_DIR / "simclr_last_checkpoint.pth"              # lets Stage 1 resume after a disconnect
UNET_PATH = PROJECT_DIR / "resnet18_unet_dental_seg_v2_dropout.pth"       # output of Stage 2 (best validation Dice)

# ---------- Settings from the paper ----------
IMAGE_SIZE = 224
SSL_EPOCHS = 100
SSL_BATCH_SIZE = 64
SSL_TEMPERATURE = 0.5
SEG_EPOCHS = 25
SEG_BATCH_SIZE = 16
DICE_THRESHOLD = 0.70    # a mask with Dice >= 0.70 counts as acceptable

# ---------- Other settings ----------
SEED = 42
NUM_WORKERS = 2
FORCE_RETRAIN = False    # True = ignore saved weights and train again

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")

## 1. Helper functions

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def list_images(folder):
    """Return the sorted image file names in a folder."""
    return sorted(p.name for p in Path(folder).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


def preprocess_xray(pil_image, size=IMAGE_SIZE):
    """CLAHE contrast enhancement and resize. Returns a 3-channel RGB image."""
    gray = np.array(pil_image.convert("L"))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    rgb = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)   # ResNet expects 3 channels
    return Image.fromarray(cv2.resize(rgb, (size, size)))


def save_figure(fig, name):
    """Save a figure as PNG and PDF (300 DPI) in RESULTS_DIR."""
    for ext in ("png", "pdf"):
        fig.savefig(RESULTS_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")

## 2. Stage 1: SimCLR pre-training (Sec. III-B)

A ResNet-18 encoder with a 2-layer projection head learns from unlabeled X-rays. Each image is turned into two random views. The NT-Xent loss (Eq. 1) pulls the two views of the same image together and pushes views of other images apart.

In [ ]:
class AddGaussianNoise:
    """Add Gaussian noise to a tensor with probability p."""

    def __init__(self, std=0.05, p=0.3):
        self.std = std
        self.p = p

    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            return tensor + torch.randn(tensor.size()) * self.std
        return tensor


class SimCLRAugmentation:
    """Turn one X-ray into two randomly augmented views (a positive pair)."""

    def __init__(self, size=IMAGE_SIZE):
        self.augment = transforms.Compose([
            transforms.RandomResizedCrop(size=size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([transforms.ColorJitter(brightness=0.3, contrast=0.3)], p=0.8),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            AddGaussianNoise(std=0.05, p=0.3),
        ])

    def __call__(self, image):
        image = preprocess_xray(image)
        return self.augment(image), self.augment(image)


class UnlabeledXrayDataset(Dataset):
    """Unlabeled X-rays for SimCLR. Each item is a pair of augmented views."""

    def __init__(self, image_dir, transform):
        self.image_dir = Path(image_dir)
        self.names = list_images(self.image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        image = Image.open(self.image_dir / self.names[idx]).convert("RGB")
        return self.transform(image)

In [ ]:
class SimCLRResNet18(nn.Module):
    """ResNet-18 encoder followed by a 2-layer projection head."""

    def __init__(self, out_dim=128):
        super().__init__()
        self.encoder = models.resnet18(weights=None)   # random start, no ImageNet weights
        feature_dim = self.encoder.fc.in_features
        self.encoder.fc = nn.Identity()                # remove the classification layer
        self.projection_head = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, out_dim),
        )

    def forward(self, x):
        return self.projection_head(self.encoder(x))


class NTXentLoss(nn.Module):
    """NT-Xent contrastive loss (Eq. 1 in the paper)."""

    def __init__(self, batch_size, temperature, device):
        super().__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.device = device
        self.criterion = nn.CrossEntropyLoss(reduction="sum")
        self.similarity = nn.CosineSimilarity(dim=-1)

    def forward(self, z_i, z_j):
        n = self.batch_size
        z = torch.cat([z_i, z_j], dim=0)                          # 2N projections
        sim = self.similarity(z.unsqueeze(1), z.unsqueeze(0))     # 2N x 2N cosine similarities

        # Column 0: similarity of the positive pair (two views of the same image).
        positives = torch.cat([torch.diag(sim, n), torch.diag(sim, -n)]).view(2 * n, 1)
        # Other columns: similarity to every other sample (the diagonal is removed).
        not_self = ~torch.eye(2 * n, dtype=torch.bool, device=self.device)
        others = sim[not_self].view(2 * n, -1)

        logits = torch.cat([positives, others], dim=1) / self.temperature
        labels = torch.zeros(2 * n, dtype=torch.long, device=self.device)   # the correct "class" is column 0
        return self.criterion(logits, labels) / (2 * n)

In [ ]:
def pretrain_simclr():
    """Train the SimCLR encoder and save its weights."""
    dataset = UnlabeledXrayDataset(UNLABELED_DIR, SimCLRAugmentation())
    print(f"Unlabeled images found: {len(dataset)}")
    # drop_last=True because the loss needs a fixed batch size.
    loader = DataLoader(dataset, batch_size=SSL_BATCH_SIZE, shuffle=True,
                        drop_last=True, num_workers=NUM_WORKERS)

    model = SimCLRResNet18().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
    criterion = NTXentLoss(SSL_BATCH_SIZE, SSL_TEMPERATURE, DEVICE)

    # Resume from the last saved checkpoint if there is one.
    start_epoch = 0
    if SSL_RESUME_PATH.exists() and not FORCE_RETRAIN:
        state = torch.load(SSL_RESUME_PATH, map_location=DEVICE)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        start_epoch = state["epoch"]
        print(f"Resuming from epoch {start_epoch}")

    for epoch in range(start_epoch, SSL_EPOCHS):
        model.train()
        total_loss = 0.0
        for view_1, view_2 in loader:
            view_1, view_2 = view_1.to(DEVICE), view_2.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(view_1), model(view_2))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1:03d}/{SSL_EPOCHS} | contrastive loss: {total_loss / len(loader):.4f}")

        # Save progress every 5 epochs, so a Colab disconnect does not lose the run.
        if (epoch + 1) % 5 == 0:
            torch.save({"model": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "epoch": epoch + 1}, SSL_RESUME_PATH)

    # Only the encoder is needed later. The projection head is dropped.
    torch.save(model.encoder.state_dict(), SSL_ENCODER_PATH)
    print(f"Encoder saved to {SSL_ENCODER_PATH}")


if SSL_ENCODER_PATH.exists() and not FORCE_RETRAIN:
    print("Stage 1 skipped: encoder weights already exist.")
else:
    pretrain_simclr()

## 3. Stage 2: U-Net segmentation (Sec. III-C)

The SimCLR encoder is used as the U-Net encoder. The model is trained with the Dice-BCE loss (Eq. 2). The checkpoint with the highest **validation** Dice is kept.

In [ ]:
class SegmentationDataset(Dataset):
    """X-ray and mask pairs. Augmentation is applied only to the training split."""

    def __init__(self, image_dir, mask_dir, train=False, size=IMAGE_SIZE):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.train = train
        self.size = size
        self.names = list_images(self.image_dir)
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __len__(self):
        return len(self.names)

    def _mask_path(self, image_name):
        """Find the mask file that has the same name as the image (.png or .jpg)."""
        stem = Path(image_name).stem
        for ext in (".png", ".jpg"):
            path = self.mask_dir / f"{stem}{ext}"
            if path.exists():
                return path
        raise FileNotFoundError(f"No mask found for {image_name}")

    def __getitem__(self, idx):
        name = self.names[idx]
        image = preprocess_xray(Image.open(self.image_dir / name).convert("RGB"), self.size)
        mask = Image.open(self._mask_path(name)).convert("L")
        mask = mask.resize((self.size, self.size), resample=Image.Resampling.NEAREST)

        # The same random flip and rotation (+-15 degrees) is applied to image and mask.
        if self.train:
            if random.random() > 0.5:
                image, mask = TF.hflip(image), TF.hflip(mask)
            if random.random() > 0.5:
                angle = random.randint(-15, 15)
                image = TF.rotate(image, angle)
                mask = TF.rotate(mask, angle, interpolation=transforms.InterpolationMode.NEAREST)

        mask = (np.array(mask) > 127).astype(np.float32)    # binary mask: 0 or 1
        image = self.normalize(transforms.ToTensor()(image))
        return image, torch.from_numpy(mask).unsqueeze(0)

In [ ]:
class DecoderBlock(nn.Module):
    """Upsample, join with the skip connection, then two 3x3 convolutions (with dropout)."""

    def __init__(self, in_channels, out_channels, dropout=0.2):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_channels * 2, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=dropout),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.upsample(x)
        if x.shape != skip.shape:    # sizes can differ by a pixel, so match them
            x = nn.functional.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=True)
        return self.conv(torch.cat([x, skip], dim=1))


class ResNet18UNet(nn.Module):
    """U-Net with a ResNet-18 encoder. Optionally starts from the SimCLR encoder weights."""

    def __init__(self, num_classes=1, ssl_weights_path=None):
        super().__init__()
        resnet = models.resnet18(weights=None)
        resnet.fc = nn.Identity()

        if ssl_weights_path is not None:
            if not Path(ssl_weights_path).exists():
                raise FileNotFoundError(f"SSL weights not found: {ssl_weights_path}")
            state = torch.load(ssl_weights_path, map_location="cpu")
            result = resnet.load_state_dict(state, strict=False)
            print(f"SSL encoder loaded ({len(result.missing_keys)} missing keys, "
                  f"{len(result.unexpected_keys)} unexpected keys)")

        # Encoder stages. Their outputs are used as skip connections.
        self.input_layer = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)   # 64 ch, 112x112
        self.pool = resnet.maxpool                                                # 64 ch, 56x56
        self.layer1 = resnet.layer1                                               # 64 ch, 56x56
        self.layer2 = resnet.layer2                                               # 128 ch, 28x28
        self.layer3 = resnet.layer3                                               # 256 ch, 14x14
        self.layer4 = resnet.layer4                                               # 512 ch, 7x7 (bottleneck)
        self.bottleneck_dropout = nn.Dropout2d(p=0.3)

        # Decoder stages
        self.decoder4 = DecoderBlock(512, 256, dropout=0.2)
        self.decoder3 = DecoderBlock(256, 128, dropout=0.2)
        self.decoder2 = DecoderBlock(128, 64, dropout=0.1)
        self.decoder1 = DecoderBlock(64, 64, dropout=0.1)
        self.final_upsample = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        init_feat = self.input_layer(x)
        x1 = self.layer1(self.pool(init_feat))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.bottleneck_dropout(self.layer4(x3))

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, init_feat)
        return self.final_conv(self.final_upsample(d1))    # raw scores (logits), 224x224

In [ ]:
class DiceBCELoss(nn.Module):
    """lambda * BCE + (1 - lambda) * Dice loss (Eq. 2 in the paper)."""

    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets, smooth=1e-6):
        probs = torch.sigmoid(logits).reshape(-1)
        targets_flat = targets.reshape(-1)
        intersection = (probs * targets_flat).sum()
        dice_loss = 1 - (2.0 * intersection + smooth) / (probs.sum() + targets_flat.sum() + smooth)
        return self.bce_weight * self.bce(logits, targets) + (1 - self.bce_weight) * dice_loss


def batch_dice(logits, targets, smooth=1e-6):
    """Dice score of a whole batch (used only to monitor training)."""
    preds = (torch.sigmoid(logits) > 0.5).float().reshape(-1)
    targets_flat = targets.reshape(-1)
    intersection = (preds * targets_flat).sum()
    return (2.0 * intersection + smooth) / (preds.sum() + targets_flat.sum() + smooth)


def train_segmentation():
    """Train the U-Net and keep the weights with the best validation Dice."""
    train_set = SegmentationDataset(LABELED_DIR / "train_images", LABELED_DIR / "train_masks", train=True)
    valid_set = SegmentationDataset(LABELED_DIR / "valid_images", LABELED_DIR / "valid_masks")
    print(f"Training pairs: {len(train_set)} | validation pairs: {len(valid_set)}")
    train_loader = DataLoader(train_set, batch_size=SEG_BATCH_SIZE, shuffle=True,
                              drop_last=True, num_workers=NUM_WORKERS)
    valid_loader = DataLoader(valid_set, batch_size=SEG_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = ResNet18UNet(ssl_weights_path=SSL_ENCODER_PATH).to(DEVICE)
    criterion = DiceBCELoss(bce_weight=0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SEG_EPOCHS)

    best_valid_dice = 0.0
    for epoch in range(SEG_EPOCHS):
        # Training pass
        model.train()
        train_loss, train_dice = 0.0, 0.0
        for images, masks in train_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_dice += batch_dice(logits, masks).item()
        scheduler.step()

        # Validation pass
        model.eval()
        valid_loss, valid_dice = 0.0, 0.0
        with torch.no_grad():
            for images, masks in valid_loader:
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                logits = model(images)
                valid_loss += criterion(logits, masks).item()
                valid_dice += batch_dice(logits, masks).item()

        train_loss, train_dice = train_loss / len(train_loader), train_dice / len(train_loader)
        valid_loss, valid_dice = valid_loss / len(valid_loader), valid_dice / len(valid_loader)
        print(f"Epoch {epoch + 1:02d}/{SEG_EPOCHS} | train loss {train_loss:.4f}, Dice {train_dice:.4f} "
              f"| valid loss {valid_loss:.4f}, Dice {valid_dice:.4f}")

        if valid_dice > best_valid_dice:
            best_valid_dice = valid_dice
            torch.save(model.state_dict(), UNET_PATH)
            print(f"  New best validation Dice. Model saved to {UNET_PATH}")


if UNET_PATH.exists() and not FORCE_RETRAIN:
    print("Stage 2 skipped: U-Net weights already exist.")
else:
    train_segmentation()

## 4. Stage 3: HUS-QA engine (Sec. III-D)

HUS-QA gives each predicted mask a quality score from 0 to 100. It only uses the model's output (the probability map), never the ground-truth mask.

| Indicator | Paper symbol | How it is computed |
|---|---|---|
| Prediction confidence | Q_c | Spread (standard deviation) of the probability map |
| Boundary sharpness | Q_s | Mean Sobel gradient of the probability map |
| Structural connectivity | Q_str | 100 for one connected region, minus 35 for each extra region |
| Anatomical area consistency | P_area | Penalty factor when the mask covers more than 25% of the image |

Final score: `Q = (0.40 * Q_c + 0.40 * Q_s + 0.20 * Q_str) * P_area` (Eq. 3 and Eq. 4).

An (almost) empty mask is handled by a separate rule, described inside the function.

The same cell also has the pixel metrics and the baseline confidence score that HUS-QA is compared with.

In [ ]:
# HUS-QA settings (the values used in the paper's experiments)
QA_WEIGHTS = {"confidence": 0.40, "sharpness": 0.40, "structure": 0.20}   # Eq. 3
CONFIDENCE_SCALE = 250.0       # converts the std of the probability map to a 0-100 score
SHARPNESS_SCALE = 500.0        # converts the mean Sobel gradient to a 0-100 score
EXTRA_REGION_PENALTY = 35.0    # points lost for each connected region after the first
AREA_LIMIT_PERCENT = 25.0      # masks covering more than this share of the image are penalised
AREA_DECAY_PERCENT = 40.0      # how fast the penalty grows above that limit
MIN_AREA_FACTOR = 0.01         # the area factor never goes below this value
EMPTY_MASK_PIXELS = 5          # fewer foreground pixels than this = "empty" mask
EMPTY_MASK_MAX_PROB = 0.05     # an empty mask whose probabilities all stay below this is a clean background


def hus_qa_score(prob):
    """HUS-QA quality score (0-100) for one predicted mask. No ground truth is used.

    prob: 2-D NumPy array with the model's foreground probabilities (values 0 to 1).
    """
    mask = prob > 0.5
    n_pixels = int(mask.sum())

    # Special rule for an (almost) empty mask: the four indicators below are not meaningful here.
    if n_pixels < EMPTY_MASK_PIXELS:
        max_prob = float(prob.max())
        if max_prob < EMPTY_MASK_MAX_PROB:
            # The model clearly sees no structure at all: high score (95 to 100).
            return 95.0 + float((1.0 - prob).mean()) * 5.0
        # Weak activations but no mask: the model probably missed something. Low score (0 to 25).
        return max(0.0, 25.0 * (1.0 - max_prob))

    # Q_c: prediction confidence (spread of the probability map)
    q_conf = min(100.0, float(prob.std(ddof=1)) * CONFIDENCE_SCALE)

    # Q_s: boundary sharpness (mean Sobel gradient magnitude of the probability map)
    grad_x = cv2.Sobel(prob, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(prob, cv2.CV_64F, 0, 1, ksize=3)
    q_sharp = min(100.0, float(np.sqrt(grad_x ** 2 + grad_y ** 2).mean()) * SHARPNESS_SCALE)

    # Q_str: structural connectivity (count connected regions in the binary mask)
    n_labels = cv2.connectedComponentsWithStats(mask.astype(np.uint8))[0]
    n_regions = n_labels - 1                                  # label 0 is the background
    q_struct = max(0.0, 100.0 - (max(n_regions, 1) - 1) * EXTRA_REGION_PENALTY)

    # P_area: penalty for masks that cover too much of the image
    area_percent = n_pixels / mask.size * 100.0
    if area_percent > AREA_LIMIT_PERCENT:
        p_area = max(MIN_AREA_FACTOR, 1.0 - (area_percent - AREA_LIMIT_PERCENT) / AREA_DECAY_PERCENT)
    else:
        p_area = 1.0

    q_geometric = (QA_WEIGHTS["confidence"] * q_conf
                   + QA_WEIGHTS["sharpness"] * q_sharp
                   + QA_WEIGHTS["structure"] * q_struct)      # Eq. 3
    return float(np.clip(q_geometric * p_area, 0.0, 100.0))   # Eq. 4


def baseline_confidence_score(prob):
    """Baseline QA: mean pixel confidence (0-100). A pixel at 0 or 1 is fully confident."""
    return float((np.abs(prob - 0.5) * 2.0).mean() * 100.0)


def segmentation_metrics(pred, target):
    """Pixel metrics for one image. pred and target are 0/1 NumPy arrays."""
    eps = 1e-6
    tp = float((pred * target).sum())
    fp = float((pred * (1 - target)).sum())
    fn = float(((1 - pred) * target).sum())
    tn = float(((1 - pred) * (1 - target)).sum())
    return {
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
        "dice": (2 * tp + eps) / (2 * tp + fp + fn + eps),
        "iou": (tp + eps) / (tp + fp + fn + eps),
        "precision": (tp + eps) / (tp + fp + eps),
        "recall": (tp + eps) / (tp + fn + eps),
    }

## 5. Evaluation on the validation and test sets (Table I)

The ground-truth masks are used here only to measure the true Dice score. HUS-QA itself never sees them.

In [ ]:
@torch.no_grad()
def evaluate(model, dataset):
    """Run the model on every image. Returns a DataFrame with one row per image."""
    model.eval()
    loader = DataLoader(dataset, batch_size=1, shuffle=False)   # HUS-QA scores one image at a time
    rows = []
    for idx, (image, mask) in enumerate(loader):
        logits = model(image.to(DEVICE))
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
        target = mask[0, 0].numpy()
        pred = (prob > 0.5).astype(np.float32)
        rows.append({
            "image": dataset.names[idx],
            "gt_pixels": int(target.sum()),
            **segmentation_metrics(pred, target),
            "baseline_qa": baseline_confidence_score(prob),
            "hus_qa": hus_qa_score(prob),
        })
    return pd.DataFrame(rows)


# Load the trained U-Net. Its weights come from UNET_PATH, so no SSL file is needed here.
model = ResNet18UNet(ssl_weights_path=None).to(DEVICE)
model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
model.eval()

valid_set = SegmentationDataset(LABELED_DIR / "valid_images", LABELED_DIR / "valid_masks")
test_set = SegmentationDataset(LABELED_DIR / "test_images", LABELED_DIR / "test_masks")
valid_df = evaluate(model, valid_set)
test_df = evaluate(model, test_set)
test_df.to_csv(RESULTS_DIR / "test_results.csv", index=False)

print(f"Table I: segmentation performance on the test set ({len(test_df)} images)")
for metric in ["accuracy", "dice", "iou", "precision", "recall"]:
    print(f"  {metric:<10}: {test_df[metric].mean():.4f}")

## 6. HUS-QA vs. the confidence baseline (Table II, Figures 2 and 3)

Two checks:
- **Pearson correlation** between the QA score and the true Dice score.
- **ROC-AUC** for finding acceptable masks (true Dice >= 0.70) using only the QA score.

In [ ]:
def safe_auc(labels, scores):
    """ROC-AUC, or NaN when only one class is present."""
    if len(np.unique(labels)) < 2:
        return float("nan")
    return roc_auc_score(labels, scores / 100.0)


def qa_statistics(qa, dice, threshold=DICE_THRESHOLD):
    """Return Pearson r, its p-value and the ROC-AUC of a QA score."""
    r, p_value = stats.pearsonr(qa, dice)
    return r, p_value, safe_auc((dice >= threshold).astype(int), qa)


def plot_qa_report(qa, dice, name, left_title, point_color, line_color, roc_label, baseline_qa=None):
    """Left: QA score vs. true Dice with the trend line. Right: ROC curve."""
    labels = (dice >= DICE_THRESHOLD).astype(int)
    r, p_value, auc = qa_statistics(qa, dice)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.scatter(qa, dice, alpha=0.5, color=point_color, s=25, edgecolors="none")
    slope, intercept = np.polyfit(qa, dice, 1)
    ax1.plot(qa, slope * qa + intercept, color=line_color, linewidth=2.5,
             label=f"Trend (r = {r:.4f}, p = {p_value:.4e})")
    ax1.set_title(left_title, fontweight="bold")
    ax1.set_xlabel("QA score (%)")
    ax1.set_ylabel("Actual Dice score")
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend(loc="upper left")

    fpr, tpr, _ = roc_curve(labels, qa / 100.0)
    ax2.plot(fpr, tpr, color="darkorange", lw=2.5, label=f"{roc_label} (AUC = {auc:.4f})")
    if baseline_qa is not None:
        b_fpr, b_tpr, _ = roc_curve(labels, baseline_qa / 100.0)
        b_auc = safe_auc(labels, baseline_qa)
        ax2.plot(b_fpr, b_tpr, color="gray", linestyle="--", lw=2, label=f"Baseline QA (AUC = {b_auc:.4f})")
    ax2.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle=":", label="Random guess")
    ax2.set_title(f"ROC curve (acceptable = Dice >= {DICE_THRESHOLD})", fontweight="bold")
    ax2.set_xlabel("False positive rate")
    ax2.set_ylabel("True positive rate")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend(loc="lower right")

    plt.tight_layout()
    save_figure(fig, name)
    plt.show()


dice = test_df["dice"].to_numpy()
baseline_qa = test_df["baseline_qa"].to_numpy()
hus_qa = test_df["hus_qa"].to_numpy()

print("Table II: quality assessment performance")
print(f"{'Method':<18}{'Pearson r':>11}{'p-value':>14}{'ROC AUC':>10}")
for method, scores in [("Baseline QA", baseline_qa), ("Proposed HUS-QA", hus_qa)]:
    r, p_value, auc = qa_statistics(scores, dice)
    print(f"{method:<18}{r:>11.4f}{p_value:>14.4e}{auc:>10.4f}")

# Validation-set check of HUS-QA (the same fixed settings, no tuning)
r_val, p_val, auc_val = qa_statistics(valid_df["hus_qa"].to_numpy(), valid_df["dice"].to_numpy())
print(f"\nHUS-QA on the validation set: r = {r_val:.4f}, p = {p_val:.4e}, AUC = {auc_val:.4f}")

# Figure 2: confidence baseline only. Figure 3: HUS-QA, with the baseline ROC for comparison.
plot_qa_report(baseline_qa, dice, "fig2_baseline_qa", "Baseline general correlation",
               "royalblue", "crimson", "Baseline QA")
plot_qa_report(hus_qa, dice, "fig3_hus_qa", "Unsupervised HUS-QA score vs. actual Dice",
               "crimson", "darkblue", "Proposed HUS-QA", baseline_qa=baseline_qa)

## 7. Threshold sensitivity (Table III, Figure 4)

The "acceptable mask" cutoff is changed from 0.60 to 0.80 to check that the result does not depend on the choice of 0.70.

In [ ]:
thresholds = [0.60, 0.65, 0.70, 0.75, 0.80]
rows = []
for th in thresholds:
    labels = (dice >= th).astype(int)
    rows.append({
        "Dice threshold": th,
        "HUS-QA AUC": round(safe_auc(labels, hus_qa), 4),
        "Baseline QA AUC": round(safe_auc(labels, baseline_qa), 4),
        "N (acceptable)": int(labels.sum()),
        "N (poor)": int(len(labels) - labels.sum()),
    })
sensitivity_df = pd.DataFrame(rows)
sensitivity_df.to_csv(RESULTS_DIR / "threshold_sensitivity.csv", index=False)
print("Table III: ROC-AUC across Dice thresholds")
print(sensitivity_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(thresholds, sensitivity_df["HUS-QA AUC"], marker="o", linewidth=2.5, color="darkorange", label="Proposed HUS-QA")
ax.plot(thresholds, sensitivity_df["Baseline QA AUC"], marker="s", linewidth=2, linestyle="--", color="gray", label="Baseline QA")
ax.axvline(DICE_THRESHOLD, color="red", linestyle=":", alpha=0.6, label=f"Threshold used in paper ({DICE_THRESHOLD})")
ax.set_xlabel("Dice threshold (good/bad cutoff)")
ax.set_ylabel("ROC-AUC")
ax.set_title("HUS-QA robustness across Dice thresholds", fontweight="bold")
ax.set_ylim([0, 1.0])
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="lower right")
plt.tight_layout()
save_figure(fig, "fig4_threshold_sensitivity")
plt.show()

## 8. Qualitative examples (Figure 5)

Three test images: the best case, the median case and the worst case. The best and median cases are chosen only among images that have a ground-truth mask.

In [ ]:
with_gt = test_df[test_df["gt_pixels"] > 0]
by_dice = with_gt["dice"].sort_values().index
selected = [
    with_gt["dice"].idxmax(),            # best case
    by_dice[len(by_dice) // 2],          # median case
    test_df["dice"].idxmin(),            # worst case (all images)
]
case_titles = ["Case A: High-Quality (Best)", "Case B: Moderate (Median)", "Case C: Poor/Failed (Worst)"]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
with torch.no_grad():
    for row, idx in enumerate(selected):
        raw = Image.open(test_set.image_dir / test_set.names[idx]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))
        image, mask = test_set[idx]
        prob = torch.sigmoid(model(image.unsqueeze(0).to(DEVICE)))[0, 0].cpu().numpy()

        axes[row, 0].imshow(raw)
        axes[row, 0].imshow(mask[0].numpy(), alpha=0.35, cmap="Greens")    # ground truth in green
        axes[row, 0].set_title(f"{case_titles[row]}\nOriginal + GT", fontsize=9, fontweight="bold")

        axes[row, 1].imshow((prob > 0.5).astype(np.float32), cmap="coolwarm")
        axes[row, 1].set_title(f"UNet Prediction\nActual Dice: {test_df.loc[idx, 'dice']:.4f}",
                               fontsize=9, fontweight="bold")

        axes[row, 2].imshow(prob, cmap="jet")
        axes[row, 2].set_title(f"Proposed HUS-QA: {test_df.loc[idx, 'hus_qa']:.2f}%\n"
                               f"Baseline QA: {test_df.loc[idx, 'baseline_qa']:.2f}%",
                               fontsize=9, fontweight="bold")
        for ax in axes[row]:
            ax.axis("off")

plt.tight_layout()
save_figure(fig, "fig5_qualitative_examples")
plt.show()